# CPSTN(Training) - Draft 4

**Purpose:** fast, graph-safe spatio-temporal traffic forecasting training pipeline for Google Colab.

### What changed from the previous notebook
- Uses **METR-LA by default (207 sensors)** instead of forcing the larger 325-node PEMS-BAY graph.
- Supports **PEMS-BAY (325 sensors)** through one configuration switch.
- Uses the benchmark road-network adjacency and **explicitly aligns sensor IDs between the HDF5 data and adjacency metadata**. The old notebook assumed matrix order matched the data columns, which is a dangerous graph-integrity assumption.
- Uses **sparse graph propagation** instead of dense `N x N` `einsum` graph multiplication.
- Uses a compact two-block STGCN-style model with residual temporal paths.
- Pre-materializes sliding windows as contiguous tensors for faster GPU training instead of repeatedly constructing NumPy windows inside `__getitem__`.
- Uses AMP mixed precision, pinned memory, persistent workers when available, gradient clipping, early stopping, and validation-only checkpoint selection.
- Includes graph integrity checks, adjacency heatmap, degree statistics, and a **sensor-coordinate graph plot** when location metadata is available.
- Includes final test evaluation and prediction export so the saved checkpoint is immediately auditable.

**Default task:** 12 historical 5-minute steps -> 12 future steps (1 hour).

In [ ]:
# 1. Install only the small set of packages that are not guaranteed by Colab.
!pip -q install gdown tables scipy scikit-learn tqdm

In [ ]:
# 2. Imports, reproducibility, device and speed settings
import os, time, math, random, pickle, json, urllib.request, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.metrics import mean_absolute_error, mean_squared_error

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("GPU is required. In Colab: Runtime -> Change runtime type -> GPU.")

device = torch.device("cuda")
torch.backends.cudnn.benchmark = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

## 3. Configuration

**Recommended first run:** leave `DATASET = "METR-LA"`. It has 207 sensors and is substantially lighter than PEMS-BAY's 325 sensors while remaining a standard graph-traffic benchmark.

Set `FAST_DEBUG = True` only to verify the entire pipeline quickly. Set it back to `False` for the real run.

In [ ]:
DATASET = "METR-LA"      # "METR-LA" or "PEMS-BAY"
FAST_DEBUG = False       # True = tiny smoke training, False = real training

INPUT_STEPS = 12         # 60 minutes of history
HORIZON = 12              # 60 minutes ahead
BATCH_SIZE = 128          # lower to 64 if your Colab GPU runs out of memory
HIDDEN = 32
DROPOUT = 0.10

MAX_EPOCHS = 25 if not FAST_DEBUG else 2
EARLY_STOP_PATIENCE = 5 if not FAST_DEBUG else 1
LEARNING_RATE = 2e-3
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 2.0

TRAIN_FRACTION = 0.70
VAL_FRACTION = 0.10

ROOT = Path("/content/cpstn_draft4")
ROOT.mkdir(parents=True, exist_ok=True)
DATA_DIR = ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)
CHECKPOINT_PATH = ROOT / f"CPSTN_Draft4_{DATASET}_best.pt"

DATA_CFG = {
    "METR-LA": {
        "h5": DATA_DIR / "metr-la.h5",
        "h5_url": "https://drive.google.com/uc?export=download&id=1pAGRfzMx6K9WWsfDcD1NMbIif0T0saFC",
        "adj": DATA_DIR / "adj_mx.pkl",
        "adj_url": "https://raw.githubusercontent.com/liyaguang/DCRNN/master/data/sensor_graph/adj_mx.pkl",
        "locations": DATA_DIR / "graph_sensor_locations.csv",
        "locations_url": "https://raw.githubusercontent.com/liyaguang/DCRNN/master/data/sensor_graph/graph_sensor_locations.csv",
        "expected_nodes": 207,
    },
    "PEMS-BAY": {
        "h5": DATA_DIR / "pems-bay.h5",
        "h5_url": "https://drive.google.com/uc?export=download&id=1wD-mHlqAb2mtHOe_68fZvDh1LpDegMMq",
        "adj": DATA_DIR / "adj_mx_bay.pkl",
        "adj_url": "https://raw.githubusercontent.com/liyaguang/DCRNN/master/data/sensor_graph/adj_mx_bay.pkl",
        "locations": DATA_DIR / "graph_sensor_locations_bay.csv",
        "locations_url": "https://raw.githubusercontent.com/liyaguang/DCRNN/master/data/sensor_graph/graph_sensor_locations_bay.csv",
        "expected_nodes": 325,
    },
}
cfg = DATA_CFG[DATASET]
print("Dataset:", DATASET, "| expected sensors:", cfg["expected_nodes"])

In [ ]:
# 4. Download benchmark data and official graph metadata
import gdown

def download_file(url, path, drive_id=None):
    path = Path(path)
    if path.exists() and path.stat().st_size > 0:
        print("Using cached:", path.name, f"({path.stat().st_size/1024**2:.1f} MB)")
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    if drive_id:
        print("Downloading from Google Drive:", path.name)
        gdown.download(id=drive_id, output=str(path), quiet=False)
    else:
        print("Downloading:", path.name)
        urllib.request.urlretrieve(url, path)
    if not path.exists() or path.stat().st_size == 0:
        raise RuntimeError(f"Download failed: {path}")

drive_id = cfg["h5_url"].split("id=")[-1]
download_file(cfg["h5_url"], cfg["h5"], drive_id=drive_id)
download_file(cfg["adj_url"], cfg["adj"])
try:
    download_file(cfg["locations_url"], cfg["locations"])
except Exception as e:
    print("Location file unavailable; graph matrix checks will still run.")
    cfg["locations"] = None

## 5. Load the HDF5 traffic table and align the graph by sensor ID

This is one of the important fixes. A graph is only meaningful if node `i` in the adjacency matrix is the same physical sensor as node `i` in the traffic tensor. The pipeline below extracts sensor IDs from the HDF5 table and from the benchmark adjacency metadata, then reorders the adjacency matrix to exactly match the data-column order.

In [ ]:
# 5. Load traffic speed data
try:
    df = pd.read_hdf(cfg["h5"], key="speed")
except (KeyError, ValueError):
    df = pd.read_hdf(cfg["h5"])

df.index = pd.to_datetime(df.index)
df = df.sort_index()
df.columns = [str(c) for c in df.columns]

if df.shape[1] != cfg["expected_nodes"]:
    raise ValueError(f"{DATASET}: expected {cfg['expected_nodes']} sensors, got {df.shape[1]}")

df = df.apply(pd.to_numeric, errors="coerce")
missing_before = int(df.isna().sum().sum())

# Traffic benchmark files commonly encode missing speed as NaN or zero.
# Keep genuine zero values out of the graph/model input by converting them to NaN,
# then interpolate only the missing observations.
df = df.replace([np.inf, -np.inf], np.nan)
df = df.replace(0.0, np.nan)
df = df.interpolate(method="linear", limit_direction="both").ffill().bfill()

if df.isna().any().any():
    raise RuntimeError("NaNs remain after interpolation.")

speed = df.to_numpy(dtype=np.float32)
timestamps = df.index

print("Data shape:", speed.shape)
print("Time range:", timestamps.min(), "->", timestamps.max())
print("Median interval:", timestamps.to_series().diff().dropna().median())
print("Missing values before repair:", missing_before)
print("Min/Max speed:", float(speed.min()), float(speed.max()))

In [ ]:
# 6. Load and align the benchmark adjacency matrix
with open(cfg["adj"], "rb") as f:
    obj = pickle.load(f, encoding="latin1")

if isinstance(obj, (tuple, list)) and len(obj) >= 3:
    graph_sensor_ids = [str(x) for x in obj[0]]
    raw_adj = np.asarray(obj[-1], dtype=np.float32)
else:
    graph_sensor_ids = None
    raw_adj = np.asarray(obj, dtype=np.float32)

if raw_adj.shape != (cfg["expected_nodes"], cfg["expected_nodes"]):
    raise ValueError(f"Adjacency shape {raw_adj.shape} does not match expected graph size.")

data_sensor_ids = list(df.columns)

if graph_sensor_ids is not None:
    graph_sensor_ids = [str(x) for x in graph_sensor_ids]
    if set(graph_sensor_ids) != set(data_sensor_ids):
        missing = sorted(set(data_sensor_ids) - set(graph_sensor_ids))[:10]
        extra = sorted(set(graph_sensor_ids) - set(data_sensor_ids))[:10]
        raise ValueError(f"Sensor-ID mismatch. Missing in graph: {missing}; extra in graph: {extra}")

    graph_pos = {sid: i for i, sid in enumerate(graph_sensor_ids)}
    order = [graph_pos[sid] for sid in data_sensor_ids]
    A = raw_adj[np.ix_(order, order)]
else:
    # Only safe when the source does not expose IDs. Keep the source matrix order.
    A = raw_adj.copy()

A = np.nan_to_num(A, nan=0.0, posinf=0.0, neginf=0.0)
A[A < 0] = 0
np.fill_diagonal(A, 1.0)

# Symmetric normalization for an STGCN-style message-passing operator.
A = np.maximum(A, A.T)
degree = A.sum(axis=1)
D_inv_sqrt = 1.0 / np.sqrt(np.maximum(degree, 1e-8))
A_norm = (D_inv_sqrt[:, None] * A * D_inv_sqrt[None, :]).astype(np.float32)

# Sparse representation: the key speed/memory fix.
coo = np.nonzero(A_norm)
vals = A_norm[coo].astype(np.float32)
indices = torch.tensor(np.vstack(coo), dtype=torch.long, device=device)
values = torch.tensor(vals, dtype=torch.float32, device=device)
A_dense = torch.sparse_coo_tensor(indices, values, size=A_norm.shape, device=device).coalesce()

print("Graph aligned to data columns:", A_norm.shape)
print("Non-zero edges incl. self-loops:", int(np.count_nonzero(A_norm)))
print("Density:", round(np.count_nonzero(A_norm) / A_norm.size, 5))
print("Degree min/mean/max:", round(float(degree.min()),2), round(float(degree.mean()),2), round(float(degree.max()),2))

assert A_norm.shape[0] == speed.shape[1]
assert np.isfinite(A_norm).all()
assert np.allclose(A_norm, A_norm.T, atol=1e-6)

In [ ]:
# 7. Graph sanity checks and visual diagnostics
plt.figure(figsize=(7, 6))
plt.imshow(A_norm, aspect="auto")
plt.title(f"{DATASET}: normalized adjacency")
plt.xlabel("Sensor index")
plt.ylabel("Sensor index")
plt.colorbar()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
plt.hist(degree, bins=30)
plt.title(f"{DATASET}: raw graph degree distribution")
plt.xlabel("Degree")
plt.ylabel("Sensors")
plt.tight_layout()
plt.show()

# Coordinate graph, if source metadata is available.
if cfg["locations"] is not None and Path(cfg["locations"]).exists():
    loc = pd.read_csv(cfg["locations"])
    loc.columns = [str(c).lower() for c in loc.columns]
    id_col = next((c for c in loc.columns if c in {"sensor_id","sensor","id"}), None)
    lat_col = next((c for c in loc.columns if "lat" in c), None)
    lon_col = next((c for c in loc.columns if "lon" in c or "lng" in c), None)

    if id_col and lat_col and lon_col:
        loc[id_col] = loc[id_col].astype(str)
        loc = loc.set_index(id_col).reindex(data_sensor_ids)
        if loc[[lat_col, lon_col]].notna().all().all():
            # Plot only actual off-diagonal graph edges to avoid drawing self-loops.
            rr, cc = np.where((A > 0) & (~np.eye(A.shape[0], dtype=bool)))
            # Avoid a huge pile of repeated lines by showing the graph edges once.
            seen = set()
            plt.figure(figsize=(9, 7))
            for i, j in zip(rr, cc):
                e = (min(i,j), max(i,j))
                if e in seen:
                    continue
                seen.add(e)
                plt.plot(
                    [loc.iloc[i][lon_col], loc.iloc[j][lon_col]],
                    [loc.iloc[i][lat_col], loc.iloc[j][lat_col]],
                    linewidth=0.35, alpha=0.12
                )
            plt.scatter(loc[lon_col], loc[lat_col], s=8)
            plt.title(f"{DATASET}: sensor road graph (aligned order)")
            plt.xlabel("Longitude")
            plt.ylabel("Latitude")
            plt.tight_layout()
            plt.show()
            print("Coordinate graph check: PASS")
        else:
            print("Location rows could not be aligned completely; skipped coordinate graph.")
    else:
        print("Location CSV schema not recognized; skipped coordinate graph.")
else:
    print("No location metadata; matrix-level graph checks completed.")

## 8. Chronological split and train-only normalization

The scaler is fitted only on the training period. Validation and test data never influence the normalization statistics.

In [ ]:
T, N = speed.shape
train_end = int(T * TRAIN_FRACTION)
val_end = int(T * (TRAIN_FRACTION + VAL_FRACTION))

train_mean = speed[:train_end].mean(axis=0).astype(np.float32)
train_std = np.maximum(speed[:train_end].std(axis=0), 1e-3).astype(np.float32)
speed_z = ((speed - train_mean) / train_std).astype(np.float32)

print("Train timestamps:", 0, "->", train_end)
print("Validation:", train_end, "->", val_end)
print("Test:", val_end, "->", T)

## 9. Fast window materialization

The previous notebook built NumPy arrays and broadcasted time features inside every `__getitem__`. That repeatedly burns CPU time during training.

Here the windows are constructed once, converted to contiguous PyTorch tensors, and then served directly by `TensorDataset`.

In [ ]:
def build_windows(speed_z, timestamps, start, end, input_steps, horizon):
    # Keep target fully inside the requested split.
    first_t = max(start + input_steps, input_steps)
    last_t = end - horizon
    if last_t < first_t:
        raise ValueError("Split is too small for the requested input/horizon.")

    starts = np.arange(first_t, last_t + 1, dtype=np.int64)

    X_speed = np.stack([speed_z[t-input_steps:t] for t in starts], axis=0)
    Y = np.stack([speed_z[t:t+horizon] for t in starts], axis=0)

    minutes = timestamps.hour.to_numpy() * 60 + timestamps.minute.to_numpy()
    tod = (minutes / 1440.0).astype(np.float32)
    tf = np.stack([np.sin(2*np.pi*tod), np.cos(2*np.pi*tod)], axis=-1).astype(np.float32)
    X_time = np.stack([tf[t-input_steps:t] for t in starts], axis=0)
    X_time = np.broadcast_to(X_time[:, :, None, :], (len(starts), input_steps, N, 2)).copy()

    X = np.concatenate([X_speed[..., None], X_time], axis=-1).astype(np.float32)
    return torch.from_numpy(np.ascontiguousarray(X)), torch.from_numpy(np.ascontiguousarray(Y))

if FAST_DEBUG:
    # Keep enough windows to exercise all code paths while making the smoke test tiny.
    dbg_train_end = min(train_end, 3000)
    dbg_val_end = min(val_end, dbg_train_end + 600)
    X_train, Y_train = build_windows(speed_z, timestamps, 0, dbg_train_end, INPUT_STEPS, HORIZON)
    X_val, Y_val = build_windows(speed_z, timestamps, dbg_train_end, dbg_val_end, INPUT_STEPS, HORIZON)
    X_test, Y_test = build_windows(speed_z, timestamps, dbg_val_end, min(T, dbg_val_end + 600), INPUT_STEPS, HORIZON)
else:
    X_train, Y_train = build_windows(speed_z, timestamps, 0, train_end, INPUT_STEPS, HORIZON)
    X_val, Y_val = build_windows(speed_z, timestamps, train_end, val_end, INPUT_STEPS, HORIZON)
    X_test, Y_test = build_windows(speed_z, timestamps, val_end, T, INPUT_STEPS, HORIZON)

print("Train:", tuple(X_train.shape), tuple(Y_train.shape))
print("Val  :", tuple(X_val.shape), tuple(Y_val.shape))
print("Test :", tuple(X_test.shape), tuple(Y_test.shape))

assert X_train.shape[1:] == (INPUT_STEPS, N, 3)
assert Y_train.shape[1:] == (HORIZON, N)

In [ ]:
# 10. DataLoaders
train_loader = DataLoader(
    TensorDataset(X_train, Y_train),
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
)
val_loader = DataLoader(
    TensorDataset(X_val, Y_val),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
)
test_loader = DataLoader(
    TensorDataset(X_test, Y_test),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
)

xb, yb = next(iter(train_loader))
print("Batch X:", tuple(xb.shape), "| Batch Y:", tuple(yb.shape))

## 11. CPSTN model
The graph layer uses dense normalized adjacency instead of CUDA sparse matrix multiplication. This avoids the `addmm_sparse_cuda` FP16 limitation while keeping the benchmark graph intact.


In [ ]:
# 11. CPSTN model - dense graph propagation for CUDA/AMP compatibility

class DenseGraphConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.proj = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x, A_dense):
        # x: [B, C, T, N]
        B, C, Tm, Nn = x.shape
        z = x.permute(0, 2, 1, 3).contiguous().reshape(B * Tm * C, Nn)
        # Keep only graph propagation in FP32. Dense CUDA matmul is AMP-safe.
        with torch.amp.autocast(device_type="cuda", enabled=False):
            z = torch.matmul(A_dense.float(), z.float().T).T
        z = z.to(dtype=x.dtype)
        z = z.reshape(B, Tm, C, Nn).permute(0, 2, 1, 3).contiguous()
        return self.proj(z)

class TemporalGate(nn.Module):
    def __init__(self, in_channels, out_channels, dilation=1, dropout=0.1):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, 2 * out_channels, kernel_size=(3, 1), padding=(dilation, 0), dilation=(dilation, 1))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        a, b = self.conv(x).chunk(2, dim=1)
        return self.dropout(torch.tanh(a) * torch.sigmoid(b))

class CPSTNBlock(nn.Module):
    def __init__(self, channels, dropout=0.1, dilation=1):
        super().__init__()
        self.temporal = TemporalGate(channels, channels, dilation=dilation, dropout=dropout)
        self.graph = DenseGraphConv(channels, channels)
        self.norm = nn.BatchNorm2d(channels)
        self.ff = nn.Conv2d(channels, channels, kernel_size=1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, A_dense):
        residual = x
        x = self.temporal(x)
        x = F.gelu(self.graph(x, A_dense))
        x = self.ff(x)
        x = self.dropout(self.norm(x))
        return F.gelu(x + residual)

class CPSTN(nn.Module):
    def __init__(self, input_channels=3, hidden=32, horizon=12, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Conv2d(input_channels, hidden, kernel_size=1)
        self.block1 = CPSTNBlock(hidden, dropout, dilation=1)
        self.block2 = CPSTNBlock(hidden, dropout, dilation=2)
        self.block3 = CPSTNBlock(hidden, dropout, dilation=4)
        self.head = nn.Sequential(
            nn.Conv2d(hidden, hidden, kernel_size=1), nn.GELU(), nn.Dropout(dropout),
            nn.Conv2d(hidden, horizon, kernel_size=1)
        )

    def forward(self, x, A_dense):
        x = x.permute(0, 3, 1, 2).contiguous()
        x = self.input_proj(x)
        x = self.block1(x, A_dense)
        x = self.block2(x, A_dense)
        x = self.block3(x, A_dense)
        x = x[:, :, -1:, :]
        return self.head(x).squeeze(2)

# Regular FP32 CUDA tensor. Tiny graph: METR-LA 207x207, PEMS-BAY 325x325.
A_dense = torch.tensor(A_norm, dtype=torch.float32, device=device)
assert A_dense.shape == (N, N)
assert torch.isfinite(A_dense).all()
print("Graph:", tuple(A_dense.shape), A_dense.dtype, A_dense.device)

model = CPSTN(input_channels=3, hidden=HIDDEN, horizon=HORIZON, dropout=DROPOUT).to(device)

# Mandatory forward smoke test before training.
model.eval()
with torch.no_grad():
    sample_out = model(xb[:2].to(device), A_dense)
assert sample_out.shape == (2, HORIZON, N)
assert torch.isfinite(sample_out).all()
params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Forward pass:", tuple(sample_out.shape))
print("Trainable parameters:", f"{params:,}")
print("Graph/Model smoke test: PASS")
model.train()


In [ ]:
# 12. Training loop with AMP + dense FP32 graph operation
criterion = nn.SmoothL1Loss(beta=0.5)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2, min_lr=1e-5)
scaler = torch.amp.GradScaler("cuda", enabled=True)

def run_epoch(loader, training):
    model.train(training)
    total = 0.0
    count = 0
    for X, Y in loader:
        X = X.to(device, non_blocking=True)
        Y = Y.to(device, non_blocking=True)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
            pred = model(X, A_dense)
            loss = criterion(pred, Y)
        if training:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
        bs = X.size(0)
        total += float(loss.detach().item()) * bs
        count += bs
    return total / max(count, 1)

best_val = float("inf")
bad_epochs = 0
history = {"train": [], "val": [], "lr": [], "epoch_seconds": []}

for epoch in range(1, MAX_EPOCHS + 1):
    start_time = time.time()
    train_loss = run_epoch(train_loader, True)
    with torch.no_grad():
        val_loss = run_epoch(val_loader, False)
    scheduler.step(val_loss)
    lr = optimizer.param_groups[0]["lr"]
    elapsed = time.time() - start_time
    history["train"].append(train_loss)
    history["val"].append(val_loss)
    history["lr"].append(lr)
    history["epoch_seconds"].append(elapsed)
    improved = val_loss < best_val
    if improved:
        best_val = val_loss
        bad_epochs = 0
        torch.save({
            "model_state_dict": model.state_dict(), "dataset": DATASET, "num_nodes": N,
            "input_steps": INPUT_STEPS, "horizon": HORIZON, "input_channels": 3,
            "hidden": HIDDEN, "dropout": DROPOUT, "mean": train_mean, "std": train_std,
            "adjacency_raw": A.astype(np.float32), "adjacency_normalized": A_norm.astype(np.float32),
            "sensor_ids": data_sensor_ids, "best_val_loss": float(best_val), "epoch": epoch,
            "model_name": "CPSTN-Draft4", "graph_backend": "dense_fp32_amp_safe"
        }, CHECKPOINT_PATH)
    else:
        bad_epochs += 1
    print(f"Epoch {epoch:02d}/{MAX_EPOCHS} | Train {train_loss:.5f} | Val {val_loss:.5f} | LR {lr:.2e} | {elapsed:.1f}s" + ("  <-- best" if improved else ""))
    if bad_epochs >= EARLY_STOP_PATIENCE:
        print("Early stopping.")
        break
print("Best validation loss:", best_val)
print("Checkpoint:", CHECKPOINT_PATH)


In [ ]:
# 13. Training curves
plt.figure(figsize=(9, 4))
plt.plot(history["train"], label="Train")
plt.plot(history["val"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("SmoothL1 loss")
plt.title(f"CPSTN Draft 4 - {DATASET}")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

print("Average epoch time:", round(float(np.mean(history["epoch_seconds"])), 2), "seconds")

## 14. Reload the best checkpoint and perform a real test evaluation

The test set is not used to select the model. This cell exists specifically to catch the kind of broken graph/test pipeline that motivated Draft 4.

In [ ]:
# 14. Reload best checkpoint
try:
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
except TypeError:
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device)

assert ckpt["dataset"] == DATASET
assert ckpt["num_nodes"] == N
assert ckpt["input_steps"] == INPUT_STEPS
assert ckpt["horizon"] == HORIZON
assert len(ckpt["sensor_ids"]) == N

model.load_state_dict(ckpt["model_state_dict"])
model.eval()

preds_z, trues_z = [], []
with torch.no_grad():
    for X, Y in test_loader:
        X = X.to(device, non_blocking=True)
        pred = model(X, A_dense).float().cpu().numpy()
        preds_z.append(pred)
        trues_z.append(Y.numpy())

pred_z = np.concatenate(preds_z, axis=0)
true_z = np.concatenate(trues_z, axis=0)

# Inverse transform: [samples, horizon, sensors]
pred = pred_z * train_std[None, None, :] + train_mean[None, None, :]
true = true_z * train_std[None, None, :] + train_mean[None, None, :]

mae = float(np.mean(np.abs(pred - true)))
rmse = float(np.sqrt(np.mean((pred - true) ** 2)))
mape = float(np.mean(np.abs((pred - true) / np.maximum(np.abs(true), 5.0))) * 100)

print(f"TEST MAE : {mae:.4f}")
print(f"TEST RMSE: {rmse:.4f}")
print(f"TEST MAPE: {mape:.2f}%")
print("Prediction shape:", pred.shape)

In [ ]:
# 15. Horizon-wise metrics: this is useful for checking whether the model collapses at longer horizons.
h_mae = np.mean(np.abs(pred - true), axis=(0,2))
h_rmse = np.sqrt(np.mean((pred - true)**2, axis=(0,2)))

plt.figure(figsize=(9, 4))
plt.plot(np.arange(1, HORIZON + 1) * 5, h_mae, marker="o", label="MAE")
plt.plot(np.arange(1, HORIZON + 1) * 5, h_rmse, marker="s", label="RMSE")
plt.xlabel("Forecast horizon (minutes)")
plt.ylabel("Error")
plt.title(f"{DATASET}: horizon-wise test error")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 16. Spatial sanity check: compare one test sample's true vs predicted speed over sensors.
sample_idx = 0
for h_idx in [0, min(5, HORIZON-1), HORIZON-1]:
    plt.figure(figsize=(10, 3.5))
    plt.plot(true[sample_idx, h_idx], label="True")
    plt.plot(pred[sample_idx, h_idx], label="Predicted", alpha=0.8)
    plt.xlabel("Sensor index (data/graph-aligned)")
    plt.ylabel("Speed")
    plt.title(f"{DATASET}: test sample {sample_idx}, +{(h_idx+1)*5} min")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# 17. Export predictions + graph metadata for the separate testing/visualization notebook
EXPORT_DIR = ROOT / "exports"
EXPORT_DIR.mkdir(exist_ok=True)

np.savez_compressed(
    EXPORT_DIR / f"{DATASET}_CPSTN_Draft4_test_predictions.npz",
    prediction=pred.astype(np.float32),
    target=true.astype(np.float32),
)

np.savez_compressed(
    EXPORT_DIR / f"{DATASET}_CPSTN_Draft4_graph.npz",
    adjacency_raw=A.astype(np.float32),
    adjacency_normalized=A_norm.astype(np.float32),
)

with open(EXPORT_DIR / f"{DATASET}_CPSTN_Draft4_metadata.json", "w") as f:
    json.dump({
        "dataset": DATASET,
        "model": "CPSTN-Draft4",
        "num_nodes": int(N),
        "input_steps": INPUT_STEPS,
        "horizon": HORIZON,
        "best_val_loss": float(best_val),
        "test_mae": mae,
        "test_rmse": rmse,
        "test_mape": mape,
        "sensor_ids": data_sensor_ids,
    }, f, indent=2)

print("Exports:")
for p in sorted(EXPORT_DIR.iterdir()):
    print(" -", p)

In [ ]:
# 18. Final integrity report
assert CHECKPOINT_PATH.exists()
assert np.isfinite(pred).all()
assert np.isfinite(true).all()
assert A.shape == (N, N)
assert A_norm.shape == (N, N)
assert np.allclose(A_norm, A_norm.T, atol=1e-6)
assert pred.shape == true.shape == (len(X_test), HORIZON, N)

print("=" * 70)
print("CPSTN DRAFT 4 INTEGRITY CHECK: PASS")
print("=" * 70)
print("Dataset              :", DATASET)
print("Sensors              :", N)
print("Input -> horizon     :", f"{INPUT_STEPS} -> {HORIZON}")
print("Best validation loss :", f"{best_val:.6f}")
print("Test MAE             :", f"{mae:.4f}")
print("Test RMSE            :", f"{rmse:.4f}")
print("Test MAPE            :", f"{mape:.2f}%")
print("Checkpoint           :", CHECKPOINT_PATH)
print("Graph matrix         :", A.shape)
print("Graph symmetry       : PASS")
print("Sensor-ID alignment  : PASS")
print("Prediction shape     :", pred.shape)
print("=" * 70)

In [ ]:
# 19. Download the trained checkpoint and exports as one ZIP
import shutil
zip_base = ROOT / f"CPSTN(Training)-Draft4_{DATASET}_outputs"
shutil.make_archive(str(zip_base), "zip", EXPORT_DIR)
shutil.copy2(CHECKPOINT_PATH, EXPORT_DIR / CHECKPOINT_PATH.name)

from google.colab import files
files.download(str(zip_base) + ".zip")
files.download(str(CHECKPOINT_PATH))